# TSP

In [1]:
import math
import numpy as np
import time

def read_instance(path):
    with open(path) as f:
        n = int(f.readline().strip())
        coords = []
        for _ in range(n):
            x, y = map(float, f.readline().split())
            coords.append((x, y))
    return n, coords


## Greedy

In [2]:
def greedy_tsp(n, coords):
    arr = np.asarray(coords, dtype=np.float64)
    x = arr[:, 0]
    y = arr[:, 1]

    visited = np.zeros(n, dtype=bool)
    tour = np.empty(n, dtype=np.int64)
    tour[0] = 0
    visited[0] = True
    cur = 0

    for i in range(1, n):
        dx = x - x[cur]
        dy = y - y[cur]
        d = dx * dx + dy * dy
        d[visited] = np.inf
        nxt = int(np.argmin(d))
        visited[nxt] = True
        tour[i] = nxt
        cur = nxt

    return tour.tolist()


def tour_length(tour, coords):
    total = 0.0
    n = len(tour)
    for i in range(n):
        x1, y1 = coords[tour[i]]
        x2, y2 = coords[tour[(i + 1) % n]]
        total += math.hypot(x1 - x2, y1 - y2)
    return total


tests = [
    "data/tsp_51_1",
    "data/tsp_100_3",
    "data/tsp_200_2",
    "data/tsp_574_1",
    "data/tsp_1889_1",
    "data/tsp_33810_1",
]

for test in tests:
    n, coords = read_instance(test)
    tour = greedy_tsp(n, coords)
    print(test, tour_length(tour, coords))

data/tsp_51_1 506.363165362846
data/tsp_100_3 25138.785452772318
data/tsp_200_2 36226.22143814145
data/tsp_574_1 47054.9474467063
data/tsp_1889_1 391470.44549188914


data/tsp_33810_1 78478867.03022148


Не проходит ни один порог. 

## Оптимизация 2-opt

2-opt исправляет главный недостаток жадника: оставшиеся хвосты и пересечения. Это позволяет нам пройти пороги. 2-opt берет маршрут жадника и пытается его распутать, если где-то рёбра пересекаются. Останаливается алгоритм тогда, когда за полный проход по всем парам не нашлось ни одного улучшения. Это значит, что мы дошли до локального оптимума: любое одиночное 2-opt улучшение уже не помогает либо на лимите по времени, который срабатывает на больших тестах (в данном случае последний тест, по прикидке отрабатывал бы где-то 3-5 часов). 2-opt исправляет парные пересечения. Но бывают ситуации, когда в туре нет ни одного пересечения рёбер, а маршрут всё равно далёк от оптимального, потому что точки распределены хитро и локально всё выглядит норм, а глобально не хватает смелых перестановок. Тогда 2-opt останавливается в локальном оптимуме. 


In [ ]:
def two_opt(tour, coords, time_limit=300):
    n = len(tour)
    if n < 4:
        return tour

    pts = np.asarray(coords, dtype=np.float64)
    xs = pts[:, 0]
    ys = pts[:, 1]

    tour = np.asarray(tour, dtype=np.int64).copy()

    start = time.time()
    improved = True

    while improved:
        improved = False
        if time.time() - start > time_limit:
            break

        for i in range(n - 1):
            if time.time() - start > time_limit:
                improved = False
                break

            a = int(tour[i])
            b = int(tour[i + 1])

            if i + 2 >= n:
                continue

            js = np.arange(i + 2, n)
            cs = tour[js]
            dds = tour[(js + 1) % n]

            d_ac  = np.sqrt((xs[a] - xs[cs])**2 + (ys[a] - ys[cs])**2)
            d_bdd = np.sqrt((xs[b] - xs[dds])**2 + (ys[b] - ys[dds])**2)
            d_ab  = math.hypot(xs[a] - xs[b], ys[a] - ys[b])
            d_cdd = np.sqrt((xs[cs] - xs[dds])**2 + (ys[cs] - ys[dds])**2)

            deltas = (d_ac + d_bdd) - (d_ab + d_cdd)

            if i == 0:
                deltas[-1] = 0.0

            neg = np.where(deltas < -1e-10)[0]
            if neg.size > 0:
                k = int(neg[0])
                j = int(js[k])
                tour[i + 1:j + 1] = tour[i + 1:j + 1][::-1]
                improved = True
                break

    return tour.tolist()

tests = [
    "data/tsp_51_1",
    "data/tsp_100_3",
    "data/tsp_200_2",
    "data/tsp_574_1",
    "data/tsp_1889_1",
    "data/tsp_33810_1",
]

for test in tests:
    n, coords = read_instance(test)

    t0 = time.time()
    tour = greedy_tsp(n, coords)
    t1 = time.time()

    limit = 600
    tour = two_opt(tour, coords, time_limit=limit)
    t2 = time.time()

    print(f"{test}  n={n}  len={tour_length(tour, coords):.2f}  "
          f"greedy={t1 - t0:.2f}s  2opt={t2 - t1:.2f}s")

data/tsp_51_1  n=51  len=440.16  greedy=0.00s  2opt=0.01s
data/tsp_100_3  n=100  len=21746.19  greedy=0.00s  2opt=0.02s
data/tsp_200_2  n=200  len=31499.54  greedy=0.00s  2opt=0.26s


data/tsp_574_1  n=574  len=39538.65  greedy=0.01s  2opt=1.22s


data/tsp_1889_1  n=1889  len=341010.66  greedy=0.03s  2opt=18.27s


data/tsp_33810_1  n=33810  len=77830903.33  greedy=5.34s  2opt=200.01s
